# Phase 2: Algorithms - Advanced Infrastructure Patterns

## Overview
Now that you understand basic infrastructure, let's implement advanced patterns used in real systems.

In this phase, you'll build:
1. Inter-agent communication protocols
2. Capability delegation systems
3. Advanced anomaly detection
4. Rollback and reversal mechanisms
5. Resource allocation

**What you'll learn:** How infrastructure handles complex scenarios

---

## Part 1: Inter-Agent Communication Protocol

Agents need a safe way to communicate with each other.

In [ ]:
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from datetime import datetime
import json

@dataclass
class Message:
    """A message passed between agents through infrastructure."""
    sender_id: str
    receiver_id: str
    message_type: str
    content: Dict[str, Any]
    timestamp: datetime = None
    message_id: str = ""
    priority: int = 1
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now()
        if not self.message_id:
            self.message_id = f"{self.sender_id}_{int(self.timestamp.timestamp())}"

class MessageBroker:
    """Infrastructure component: Inter-agent communication."""
    
    def __init__(self):
        self.message_queue: List[Message] = []
        self.agent_mailboxes: Dict[str, List[Message]] = {}
        self.message_history: List[Message] = []
    
    def send_message(self, message: Message) -> bool:
        """Send a message between agents."""
        # Validate
        if not self._validate_message(message):
            return False
        
        # Add to queue and mailbox
        self.message_queue.append(message)
        
        if message.receiver_id not in self.agent_mailboxes:
            self.agent_mailboxes[message.receiver_id] = []
        self.agent_mailboxes[message.receiver_id].append(message)
        
        # Log for history
        self.message_history.append(message)
        
        print(f"✓ Message from {message.sender_id} to {message.receiver_id}")
        return True
    
    def _validate_message(self, message: Message) -> bool:
        """Validate message before delivery."""
        # Check required fields
        if not message.sender_id or not message.receiver_id:
            return False
        
        # Could add more validation
        return True
    
    def get_messages(self, agent_id: str) -> List[Message]:
        """Get all messages for an agent."""
        return self.agent_mailboxes.get(agent_id, [])
    
    def process_message(self, agent_id: str) -> Optional[Message]:
        """Process next message for an agent."""
        if agent_id in self.agent_mailboxes and self.agent_mailboxes[agent_id]:
            # Get highest priority message
            messages = self.agent_mailboxes[agent_id]
            messages.sort(key=lambda m: m.priority, reverse=True)
            return messages.pop(0)
        return None

# Test messaging
broker = MessageBroker()

msg1 = Message(
    sender_id="agent_A",
    receiver_id="agent_B",
    message_type="request_data",
    content={"query": "sales_data", "date_range": "2025-01-01 to 2025-06-01"}
)

msg2 = Message(
    sender_id="agent_B",
    receiver_id="agent_A",
    message_type="response_data",
    content={"data": [1, 2, 3, 4, 5], "rows": 5}
)

broker.send_message(msg1)
broker.send_message(msg2)

print(f"\nMessages for agent_B: {len(broker.get_messages('agent_B'))}")
print(f"Messages for agent_A: {len(broker.get_messages('agent_A'))}")

## Part 2: Rollback and Reversal

Infrastructure can undo agent actions if they cause harm.

In [ ]:
class Transaction:
    """Represents a reversible agent action."""
    def __init__(self, txn_id: str, agent_id: str, action: str, state: Dict[str, Any]):
        self.txn_id = txn_id
        self.agent_id = agent_id
        self.action = action
        self.state_before = state.copy()
        self.state_after = None
        self.timestamp = datetime.now()
        self.reversible = True

class RollbackSystem:
    """Infrastructure component: Undo harmful actions."""
    
    def __init__(self):
        self.transactions: Dict[str, Transaction] = {}
        self.transaction_log: List[Transaction] = []
    
    def start_transaction(self, agent_id: str, action: str, 
                         state: Dict[str, Any]) -> str:
        """Start tracking a reversible action."""
        txn_id = f"{agent_id}_{len(self.transactions)}"
        txn = Transaction(txn_id, agent_id, action, state)
        self.transactions[txn_id] = txn
        self.transaction_log.append(txn)
        return txn_id
    
    def commit_transaction(self, txn_id: str, new_state: Dict[str, Any]):
        """Finalize a transaction."""
        if txn_id in self.transactions:
            self.transactions[txn_id].state_after = new_state
            print(f"✓ Committed {txn_id}")
    
    def rollback_transaction(self, txn_id: str) -> bool:
        """Undo a transaction."""
        if txn_id not in self.transactions:
            return False
        
        txn = self.transactions[txn_id]
        if not txn.reversible:
            print(f"✗ Cannot rollback {txn_id} (irreversible action)")
            return False
        
        print(f"✓ Rolled back {txn_id}")
        print(f"  Action: {txn.action}")
        print(f"  Before: {txn.state_before}")
        print(f"  After: {txn.state_after}")
        return True

# Test rollback
rollback = RollbackSystem()

# Simulate purchase transaction
state_before = {"balance": 1000}
txn1 = rollback.start_transaction("agent_1", "purchase", state_before)
state_after = {"balance": 800}  # After $200 purchase
rollback.commit_transaction(txn1, state_after)

print("\nAttempting to reverse purchase...")
rollback.rollback_transaction(txn1)

## Part 3: Advanced Anomaly Detection

Machine learning approaches to detect unusual agent behavior.

In [ ]:
from statistics import mean, stdev

class AnomalyDetector:
    """Infrastructure component: Advanced anomaly detection."""
    
    def __init__(self):
        self.agent_behavior: Dict[str, List[Dict[str, Any]]] = {}
        self.anomalies: List[Dict[str, Any]] = []
    
    def record_behavior(self, agent_id: str, metric_name: str, value: float):
        """Record agent behavior metric."""
        if agent_id not in self.agent_behavior:
            self.agent_behavior[agent_id] = []
        
        self.agent_behavior[agent_id].append({
            "metric": metric_name,
            "value": value,
            "timestamp": datetime.now()
        })
    
    def detect_outliers(self, agent_id: str, metric: str, 
                       threshold: float = 2.0) -> List[float]:
        """Detect values more than N standard deviations from mean."""
        if agent_id not in self.agent_behavior:
            return []
        
        values = [
            b["value"] for b in self.agent_behavior[agent_id]
            if b["metric"] == metric
        ]
        
        if len(values) < 2:
            return []
        
        avg = mean(values)
        std = stdev(values)
        
        outliers = [
            v for v in values
            if abs(v - avg) > threshold * std
        ]
        
        return outliers
    
    def report_anomalies(self):
        """Print detected anomalies."""
        print("\n=== Anomaly Report ===")
        for agent_id in self.agent_behavior:
            outliers = self.detect_outliers(agent_id, "transaction_amount", 2.0)
            if outliers:
                print(f"{agent_id}: Found {len(outliers)} unusual transactions")
                for val in outliers:
                    print(f"  - ${val}")

# Test anomaly detection
detector = AnomalyDetector()

# Record normal transactions
for amount in [100, 105, 95, 110, 98, 102]:
    detector.record_behavior("agent_X", "transaction_amount", amount)

# Record unusual transaction
detector.record_behavior("agent_X", "transaction_amount", 5000)

detector.report_anomalies()

## Summary: Advanced Patterns

In this phase, you implemented:

1. ✅ **Message Broker** - Safe inter-agent communication
2. ✅ **Rollback System** - Undo harmful actions
3. ✅ **Anomaly Detector** - Detect unusual behavior

These patterns enable sophisticated infrastructure.

---

## Next Step

Ready for real-world applications? Move to **Phase 3: Applications**

You'll apply infrastructure to:
- Multi-agent coordination
- Governance systems
- Impact assessment
- Production deployment